## WP006 — Per-Team Sigma (partial pooling)

WP005 found that loosening the *global* shrinkage priors — one shared `init_scale`, `home_adv_sd`, `sigma_att/def` for every team — doesn't recover resolution, individually or combined, though the combined arm (`loose_combo`) showed a small, consistent-but-not-significant improvement across three separate tests.

This work product tests a structurally different mechanism: **let each team's own AR(1) innovation SD be partially pooled toward a shared population value, instead of forcing every team through one global sigma.** A promoted side or a team mid-managerial-change plausibly needs more round-to-round volatility than a stable top-six squad — a global knob can't represent that; a global bump (WP005) helps and hurts different teams simultaneously, which is a coherent explanation for why it read flat.

New model code (not a config override): `ar1_hierarchical_sigma` in `src/football_model/model/priors.py`, wired into `build_model` via `ModelConfig.use_per_team_sigma`. Non-centered (a HalfNormal population scale, each team's own sigma = a fixed HalfNormal(1) draw × that scale — the same funnel-avoidance pattern as `home_adv`). Fully covered by new tests in `test_priors.py`, `test_components.py`, `test_model.py` before any of the results below were run.

**Four arms**, reusing WP001/WP005 checkpoints where possible (no compute wasted re-running configs already fit):

| arm | `use_per_team_sigma` | `init_scale` | `home_adv_sd` | `sigma_att`/`def` (or pop scale) | source |
|---|---|---|---|---|---|
| `baseline` | ❌ | 0.20 | 0.02 | 0.008 | WP001's checkpoint, reused |
| `loose_combo` | ❌ | 0.30 | 0.06 | 0.020 | WP005's checkpoint, reused |
| `per_team_sigma` | ✅ | 0.20 | 0.02 | 0.008 (pop scale) | new — isolates partial pooling alone |
| `per_team_sigma_loose` | ✅ | 0.30 | 0.06 | 0.020 (pop scale) | new — pooling + WP005's other loosening combined |

Same yardstick as WP003/WP004/WP005 throughout: pooled RPS and paired-bootstrap gap to Pinnacle closing odds on the 401-match walk-forward comparison, plus the disagreement-decile overfitting check. New diagnostic specific to this WP: does each team's *fitted* `sigma_att_team`/`sigma_def_team` actually land higher on teams you'd expect (promoted sides, teams with a big mid-season swing) — checked against WP003's promoted-team subset directly.

In [1]:
import json
import pickle
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pymc as pm

from football_model.data.prepare_model_data import prepare_model_data
from football_model.model.model import build_model
from football_model.model.predict import dc_outcome_probs
from football_model.types.model_data import ModelConfig

REPO = Path('/Users/hadiahmed/Documents/projects/football-predictor')
WP001 = REPO / 'work_products' / 'wp001_walkforward_cv_baseline'
WP003 = REPO / 'work_products' / 'wp003_bookmaker_benchmark'
WP005 = REPO / 'work_products' / 'wp005_prior_loosening'
WP006 = REPO / 'work_products' / 'wp006_per_team_sigma'
SCRIPT = REPO / 'scripts' / 'run_cv_window.py'

with open(WP001 / 'cv_shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)
df_cv, windows = shared['df_cv'], shared['windows']
print(len(windows), 'windows;', df_cv['team'].nunique(), 'teams total')

35 windows; 28 teams total


### Candidate configs

In [2]:
LOOSE = dict(init_scale=0.30, home_adv_sd=0.06, sigma_att=0.020, sigma_def=0.020)  # WP005's loose_combo

ARMS = {
    'baseline':             {},
    'loose_combo':          dict(LOOSE),
    'per_team_sigma':       {'use_per_team_sigma': True},
    'per_team_sigma_loose': {'use_per_team_sigma': True, **LOOSE},
}
BASE = dict(clip_theta=5.0, center_team_strength=False, use_dixon_coles=True, use_xG=True)
for name, ov in ARMS.items():
    ModelConfig(**BASE, **ov)  # smoke: every override is a real field
print('all', len(ARMS), 'arm configs valid')

all 4 arm configs valid


## Phase 1 — Prior-predictive triage

Same idea as WP005 Phase 1, plus one new thing worth checking specifically: does `ar1_hierarchical_sigma`'s prior actually imply a *plausible spread of team-level sigma* — some teams noticeably more volatile than others — rather than either (a) collapsing back to effectively one shared value, or (b) implying wild, unrealistic differences?

In [3]:
train_data = prepare_model_data(df_cv, max_round=windows[-1]['train_end'])
N_PP = 120

def prior_predictive_summary(overrides, draws=N_PP, seed=0):
    cfg = ModelConfig(**BASE, **overrides)
    with build_model(train_data, cfg):
        idata = pm.sample_prior_predictive(draws=draws, random_seed=seed)
    pr = idata.prior
    attack = pr['attack'].values[0]
    defence = pr['defence'].values[0]
    home_adv = pr['home_adv'].values[0]
    t = attack.shape[1] - 1
    out = {
        'attack SD (cross-team)': attack[:, t, :].std(axis=1),
        'home_adv SD (cross-team)': home_adv.std(axis=1),
        'top/bottom home-goals ratio': np.exp(attack[:, t, :].max(axis=1) - defence[:, t, :].min(axis=1)),
    }
    if 'sigma_att_team' in pr:
        sat = pr['sigma_att_team'].values[0]   # (draws, n_teams)
        # within-draw spread of team sigma (how differently-volatile are teams, per draw)
        out['sigma_att_team: within-draw max/min ratio'] = sat.max(axis=1) / np.maximum(sat.min(axis=1), 1e-6)
        out['sigma_att_team: pooled mean'] = sat.mean(axis=1)
    return out

rows = []
for name, ov in ARMS.items():
    s = prior_predictive_summary(ov)
    row = {'arm': name}
    for k, v in s.items():
        row[k] = f'{np.median(v):.3f} [{np.percentile(v,5):.3f}, {np.percentile(v,95):.3f}]'
    rows.append(row)
    print(name, 'done')
pp = pd.DataFrame(rows).set_index('arm')
pd.set_option('display.max_colwidth', None); pd.set_option('display.width', 200)
pp.T

/var/folders/_7/_hqwtk652491lqydm38b4z2r0000gp/T/ipykernel_11792/3369964200.py:7: UserWarning: The effect of Potentials on other parameters is ignored during prior predictive sampling. This is likely to lead to invalid or biased predictive samples.
  idata = pm.sample_prior_predictive(draws=draws, random_seed=seed)
Sampling: [att_0, att_rw_std, beta_xG, def_0, def_rw_std, goals_away, goals_home, home_adv_raw, home_mu, home_sd, rho_att, rho_dc, rho_def, sigma_att, sigma_def]
/var/folders/_7/_hqwtk652491lqydm38b4z2r0000gp/T/ipykernel_11792/3369964200.py:7: UserWarning: The effect of Potentials on other parameters is ignored during prior predictive sampling. This is likely to lead to invalid or biased predictive samples.
  idata = pm.sample_prior_predictive(draws=draws, random_seed=seed)


baseline done


Sampling: [att_0, att_rw_std, beta_xG, def_0, def_rw_std, goals_away, goals_home, home_adv_raw, home_mu, home_sd, rho_att, rho_dc, rho_def, sigma_att, sigma_def]
/var/folders/_7/_hqwtk652491lqydm38b4z2r0000gp/T/ipykernel_11792/3369964200.py:7: UserWarning: The effect of Potentials on other parameters is ignored during prior predictive sampling. This is likely to lead to invalid or biased predictive samples.
  idata = pm.sample_prior_predictive(draws=draws, random_seed=seed)


loose_combo done


Sampling: [att_0, att_rw_std, beta_xG, def_0, def_rw_std, goals_away, goals_home, home_adv_raw, home_mu, home_sd, rho_att, rho_dc, rho_def, sigma_att_pop, sigma_att_team_raw, sigma_def_pop, sigma_def_team_raw]
/var/folders/_7/_hqwtk652491lqydm38b4z2r0000gp/T/ipykernel_11792/3369964200.py:7: UserWarning: The effect of Potentials on other parameters is ignored during prior predictive sampling. This is likely to lead to invalid or biased predictive samples.
  idata = pm.sample_prior_predictive(draws=draws, random_seed=seed)


per_team_sigma done


Sampling: [att_0, att_rw_std, beta_xG, def_0, def_rw_std, goals_away, goals_home, home_adv_raw, home_mu, home_sd, rho_att, rho_dc, rho_def, sigma_att_pop, sigma_att_team_raw, sigma_def_pop, sigma_def_team_raw]


per_team_sigma_loose done


arm,baseline,loose_combo,per_team_sigma,per_team_sigma_loose
attack SD (cross-team),"0.213 [0.163, 0.289]","0.323 [0.251, 0.460]","0.212 [0.169, 0.281]","0.322 [0.254, 0.421]"
home_adv SD (cross-team),"0.011 [0.001, 0.036]","0.034 [0.003, 0.108]","0.012 [0.002, 0.038]","0.037 [0.005, 0.115]"
top/bottom home-goals ratio,"2.352 [1.899, 3.410]","3.685 [2.598, 6.656]","2.392 [1.924, 3.472]","3.893 [2.674, 7.616]"
sigma_att_team: within-draw max/min ratio,NaN,NaN,"71.431 [19.984, 1158.142]","71.431 [19.984, 1158.142]"
sigma_att_team: pooled mean,NaN,NaN,"0.004 [0.000, 0.014]","0.009 [0.001, 0.035]"


## Phase 2 — Screening CV (reduced windows)

Every 2nd window (18 of 35), same harness as WP005 — `run_cv_window.py --config-json` already supports `use_per_team_sigma` with no script changes needed (it's just another `ModelConfig` field). `baseline`/`loose_combo` are seeded from WP001/WP005's finished checkpoints filtered to the screening windows; only the 2 new arms actually run (~36 fits).

**Heavy compute — run this yourself.**

In [4]:
SCREEN_WINDOWS = list(range(1, len(windows) + 1, 2))
DATA_PATH = WP001 / 'cv_shared_data.pkl'
WINDOW_TIMEOUT = 1200

def load_ckpt(p):
    return pickle.load(open(p, 'rb')) if p.exists() else {'results': [], 'cv_match_predictions': []}

def seed_from(src_checkpoint_path, dest_name):
    dest = WP006 / f'cv_checkpoint_{dest_name}.pkl'
    if dest.exists():
        return
    src = load_ckpt(src_checkpoint_path)
    filt = {'results': [r for r in src['results'] if r['window'] in SCREEN_WINDOWS],
            'cv_match_predictions': [m for m in src['cv_match_predictions'] if m['window'] in SCREEN_WINDOWS]}
    pickle.dump(filt, open(dest, 'wb'))
    print(f'seeded {dest_name} from {src_checkpoint_path.name}:', len(filt['results']), 'windows')

seed_from(WP001 / 'cv_checkpoint.pkl', 'baseline')
seed_from(WP005 / 'cv_checkpoint_full_loose_combo.pkl', 'loose_combo')

for arm, ov in ARMS.items():
    if arm in ('baseline', 'loose_combo'):
        continue
    ckpt = WP006 / f'cv_checkpoint_{arm}.pkl'
    done = {r['window'] for r in load_ckpt(ckpt)['results']}
    todo = [w for w in SCREEN_WINDOWS if w not in done]
    print(f'\n### arm {arm}  overrides={ov}  ({len(done)}/{len(SCREEN_WINDOWS)} done, {len(todo)} to run)')
    for w in todo:
        print(f'  [{arm}] window {w}')
        try:
            subprocess.run(
                [sys.executable, str(SCRIPT), '--data-path', str(DATA_PATH),
                 '--checkpoint-path', str(ckpt), '--window-index', str(w),
                 '--config-json', json.dumps(ov)],
                timeout=WINDOW_TIMEOUT, check=True,
            )
        except subprocess.TimeoutExpired:
            print(f'  [{arm}] window {w} TIMEOUT — skipped, re-run cell to retry')
        except subprocess.CalledProcessError:
            print(f'  [{arm}] window {w} FAILED — skipped, re-run cell to retry')

print('\nscreening run complete')
for arm in ARMS:
    n = len(load_ckpt(WP006 / f'cv_checkpoint_{arm}.pkl')['results'])
    print(f'  {arm}: {n}/{len(SCREEN_WINDOWS)}')

seeded baseline from cv_checkpoint.pkl: 18 windows
seeded loose_combo from cv_checkpoint_full_loose_combo.pkl: 18 windows

### arm per_team_sigma  overrides={'use_per_team_sigma': True}  (0/18 done, 18 to run)
  [per_team_sigma] window 1
[window 1/35] training rounds 1-36 (use_xg=True, use_dc=True, overrides={'use_per_team_sigma': True})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 0:  10%|█         | 400/4000 [00:07<00:49, 73.33it/s]

Running chain 1:  10%|█         | 400/4000 [00:09<00:58, 61.47it/s]

Running chain 0:  20%|██        | 800/4000 [00:10<00:28, 113.19it/s][A

Running chain 0:  25%|██▌       | 1000/4000 [00:11<00:22, 130.70it/s][A

Running chain 1:  25%|██▌       | 1000/4000 [00:12<00:24, 121.79it/s]

Running chain 0:  35%|███▌      | 1400/4000 [00:13<00:17, 150.74it/s]

Running chain 0:  40%|████      | 1600/4000 [00:15<00:15, 153.46it/s]

Running chain 0:  45%|████▌     | 1800/4000 [00:16<00:13, 161.60it/s]

Running chain 1:  45%|████▌     | 1800/4000 [00:17<00:14, 154.72it/s]

Running chain 0:  55%|█████▌    | 2200/4000 [00:18<00:09, 183.20it/s]

Running chain 0:  65%|██████▌   | 2600/4000 [00:19<00:06, 206.94it/s]

Running chain 1:  60%|██████    | 2400/4000 [00:21<00:11, 142.2

[window 1] MAE=1.148 LL_improvement=1.13
[window 1] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp006_per_team_sigma/cv_checkpoint_per_team_sigma.pkl
  [per_team_sigma] window 3
[window 3/35] training rounds 1-46 (use_xg=True, use_dc=True, overrides={'use_per_team_sigma': True})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 2:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 1:   5%|▌         | 200/4000 [00:09<02:15, 28.03it/s]

Running chain 1:  10%|█         | 400/4000 [00:10<01:11, 50.02it/s]

Running chain 1:  15%|█▌        | 600/4000 [00:12<00:50, 67.59it/s]

Running chain 1:  20%|██        | 800/4000 [00:14<00:40, 78.96it/s]

Running chain 0:  25%|██▌       | 1000/4000 [00:15<00:30, 96.85it/s]

Running chain 0:  30%|███       | 1200/4000 [00:17<00:26, 104.78it/s][A

Running chain 0:  35%|███▌      | 1400/4000 [00:18<00:24, 105.46it/s][A

Running chain 1:  35%|███▌      | 1400/4000 [00:19<00:24, 104.62it/s]

Running chain 1:  40%|████      | 1600/4000 [00:21<00:22, 106.61it/s]

Running chain 1:  45%|████▌     | 1800/4000 [00:23<00:20, 108.29it/s]

Running chain 0:  50%|█████     | 2000/4000 [00:24<00:18, 110.79it/s]

Running 

[window 3] MAE=0.669 LL_improvement=4.32
[window 3] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp006_per_team_sigma/cv_checkpoint_per_team_sigma.pkl
  [per_team_sigma] window 5
[window 5/35] training rounds 1-56 (use_xg=True, use_dc=True, overrides={'use_per_team_sigma': True})


...

[window 31] MAE=1.071 LL_improvement=1.23
[window 31] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp006_per_team_sigma/cv_checkpoint_per_team_sigma_loose.pkl
  [per_team_sigma_loose] window 33
[window 33/35] training rounds 1-196 (use_xg=True, use_dc=True, overrides={'use_per_team_sigma': True, 'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 0:   5%|▌         | 200/4000 [00:36<10:48,  5.86it/s]

Running chain 1:  10%|█         | 400/4000 [00:51<06:37,  9.06it/s]

Running chain 1:  15%|█▌        | 600/4000 [01:00<04:31, 12.51it/s]

Running chain 1:  20%|██        | 800/4000 [01:09<03:29, 15.29it/s]

Running chain 1:  25%|██▌       | 1000/4000 [01:17<02:52, 17.38it/s]

Running chain 1:  30%|███       | 1200/4000 [01:26<02:26, 19.13it/s]

Running chain 1:  35%|███▌      | 1400/4000 [01:34<02:08, 20.29it/s]

Running chain 1:  40%|████      | 1600/4000 [01:43<01:53, 21.15it/s]

Running chain 1:  45%|████▌     | 1800/4000 [01:51<01:40, 21.85it/s]

Running chain 1:  50%|█████     | 2000/4000 [02:00<01:30, 22.14it/s]

Running chain 1:  55%|█████▌    | 2200/4000 [02:08<01:19, 22.77it/s]

Running chain 1:  60%|██████    | 2400/4000 [02:17<01:08, 23.25it/s]

Running

[window 33] MAE=0.845 LL_improvement=-0.04
[window 33] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp006_per_team_sigma/cv_checkpoint_per_team_sigma_loose.pkl
  [per_team_sigma_loose] window 35
[window 35/35] training rounds 1-206 (use_xg=True, use_dc=True, overrides={'use_per_team_sigma': True, 'init_scale': 0.3, 'home_adv_sd': 0.06, 'sigma_att': 0.02, 'sigma_def': 0.02})


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 1:   5%|▌         | 200/4000 [00:39<11:55,  5.31it/s]

Running chain 1:  10%|█         | 400/4000 [00:49<06:26,  9.31it/s]

Running chain 1:  15%|█▌        | 600/4000 [00:58<04:29, 12.60it/s]

Running chain 1:  20%|██        | 800/4000 [01:07<03:29, 15.26it/s]

Running chain 1:  25%|██▌       | 1000/4000 [01:16<02:55, 17.09it/s]

Running chain 1:  30%|███       | 1200/4000 [01:25<02:29, 18.67it/s]

Running chain 1:  35%|███▌      | 1400/4000 [01:35<02:13, 19.48it/s]

Running chain 1:  40%|████      | 1600/4000 [01:43<01:57, 20.46it/s]

Running chain 1:  45%|████▌     | 1800/4000 [01:52<01:43, 21.18it/s]

Running chain 1:  50%|█████     | 2000/4000 [02:02<01:34, 21.12it/s]

Running chain 1:  55%|█████▌    | 2200/4000 [02:10<01:23, 21.64it/s]

Running chain 1:  60%|██████    | 2400/4000 [02:19<01:12, 22.04it/s]

Running

[window 35] MAE=0.894 LL_improvement=0.87
[window 35] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp006_per_team_sigma/cv_checkpoint_per_team_sigma_loose.pkl

screening run complete
  baseline: 18/18
  loose_combo: 18/18
  per_team_sigma: 18/18
  per_team_sigma_loose: 18/18


### Phase 2 analysis — resolution + gap to Pinnacle per arm

In [5]:
odds_raw = pd.read_pickle(WP003 / 'odds_raw.pkl')
CODE_TO_FD = {'ARS': 'Arsenal', 'AVL': 'Aston Villa', 'BOU': 'Bournemouth', 'BRE': 'Brentford',
    'BRI': 'Brighton', 'BUR': 'Burnley', 'CHE': 'Chelsea', 'CRY': 'Crystal Palace', 'EVE': 'Everton',
    'FLH': 'Fulham', 'IPS': 'Ipswich', 'LED': 'Leeds', 'LEI': 'Leicester', 'LIV': 'Liverpool',
    'LUT': 'Luton', 'MCI': 'Man City', 'MUN': 'Man United', 'NEW': 'Newcastle', 'NOR': 'Norwich',
    'NOT': "Nott'm Forest", 'SHE': 'Sheffield United', 'SOU': 'Southampton', 'SUN': 'Sunderland',
    'TOT': 'Tottenham', 'WAT': 'Watford', 'WBA': 'West Brom', 'WHU': 'West Ham', 'WOL': 'Wolves'}
df_sorted = df_cv.sort_values('datetime').reset_index(drop=True)

def fixtures_for(ckpt, windows_list=None):
    windows_list = windows if windows_list is None else windows_list
    mp = ckpt['cv_match_predictions']
    rows = []
    for w in sorted({m['window'] for m in mp}):
        win = windows_list[w - 1]
        sel = df_sorted[(df_sorted['is_home'] == 1) & (df_sorted['round'] >= win['test_start'])
                        & (df_sorted['round'] <= win['test_end'])]
        for (_, r), m in zip(sel.iterrows(), [x for x in mp if x['window'] == w]):
            assert int(r['goals_home']) == m['goals_home'] and int(r['goals_away']) == m['goals_away']
            rows.append({'date': pd.Timestamp(r['datetime']).normalize(),
                         'home_fd': CODE_TO_FD[r['team']], 'away_fd': CODE_TO_FD[r['opp_team']],
                         'lambda_home': m['lambda_home'], 'lambda_away': m['lambda_away'],
                         'rho_dc': m.get('rho_dc'), 'goals_home': m['goals_home'], 'goals_away': m['goals_away']})
    d = pd.DataFrame(rows)
    P = np.array([dc_outcome_probs(x.lambda_home, x.lambda_away, rho=x.rho_dc) for x in d.itertuples()])
    d[['p_home', 'p_draw', 'p_away']] = P
    d['result'] = np.where(d['goals_home'] > d['goals_away'], 'H',
                    np.where(d['goals_home'] == d['goals_away'], 'D', 'A'))
    return d.merge(odds_raw, left_on=['date', 'home_fd', 'away_fd'],
                   right_on=['Date', 'HomeTeam', 'AwayTeam'], how='inner')

def rps_row(ph, pdw, pa, a):
    c1, c2 = ph, ph + pdw
    e1 = 1.0 if a == 'H' else 0.0
    e2 = 1.0 if a in ('H', 'D') else 0.0
    return 0.5 * ((c1 - e1) ** 2 + (c2 - e2) ** 2)

def boot(v, n=4000, seed=0):
    v = np.asarray(v, float); rng = np.random.default_rng(seed)
    b = np.array([rng.choice(v, len(v), replace=True).mean() for _ in range(n)])
    return v.mean(), *np.percentile(b, [2.5, 97.5])

def devig(o):
    inv = 1 / np.asarray(o, float); return inv / inv.sum()

rows = []
for arm in ARMS:
    ck = WP006 / f'cv_checkpoint_{arm}.pkl'
    if not ck.exists() or len(load_ckpt(ck)['results']) == 0:
        print(f'{arm}: not run yet'); continue
    d = fixtures_for(load_ckpt(ck))
    pin_ok = d[['PSCH', 'PSCD', 'PSCA']].notna().all(axis=1) if 'PSCH' in d else pd.Series(False, index=d.index)
    d = d[pin_ok].copy()
    Pin = np.array([devig(r) for r in d[['PSCH', 'PSCD', 'PSCA']].to_numpy()])
    d['rps_m'] = [rps_row(x.p_home, x.p_draw, x.p_away, x.result) for x in d.itertuples()]
    d['rps_p'] = [rps_row(Pin[i, 0], Pin[i, 1], Pin[i, 2], d['result'].iloc[i]) for i in range(len(d))]
    d['disag'] = np.abs(d['p_home'].values - Pin[:, 0])
    g_all, lo, hi = boot((d['rps_m'] - d['rps_p']).values)
    top = d[d['disag'] >= d['disag'].quantile(0.75)]
    g_top, _, _ = boot((top['rps_m'] - top['rps_p']).values)
    rows.append({'arm': arm, 'n': len(d), 'model RPS': round(d['rps_m'].mean(), 4),
                 'gap vs Pinnacle': f'{g_all:+.4f}', 'CI': f'[{lo:+.4f},{hi:+.4f}]',
                 'gap top-25% disagree': f'{g_top:+.4f}'})
print(pd.DataFrame(rows).to_string(index=False))

                 arm   n  model RPS gap vs Pinnacle                CI gap top-25% disagree
            baseline 195     0.1916         +0.0129 [+0.0058,+0.0200]              +0.0455
         loose_combo 195     0.1905         +0.0118 [+0.0049,+0.0188]              +0.0400
      per_team_sigma 195     0.1923         +0.0136 [+0.0063,+0.0209]              +0.0450
per_team_sigma_loose 195     0.1908         +0.0121 [+0.0050,+0.0194]              +0.0423


### Phase 2b — does fitted `sigma_att_team`/`sigma_def_team` land on the teams you'd expect?

The actual point of this WP, not just the score. Compare each team's mean fitted `sigma_att_team` (averaged across screening windows where they were active) to whether they were promoted that season. If partial pooling is doing something real, promoted sides should show up with higher fitted sigma more often than chance.

In [6]:
def team_sigma_table(arm):
    ck = WP006 / f'cv_checkpoint_{arm}.pkl'
    cp = load_ckpt(ck)
    if not cp['results'] or cp['results'][0].get('sigma_att_team') is None:
        print(f'{arm}: no per-team sigma recorded (use_per_team_sigma=False, or not run yet)')
        return None
    rows = []
    for r in cp['results']:
        for team, val in r['sigma_att_team'].items():
            rows.append({'team': team, 'window': r['window'], 'sigma_att_team': val})
    d = pd.DataFrame(rows)
    return d.groupby('team')['sigma_att_team'].mean().sort_values(ascending=False)

season_teams = {s: set(g['team']) | set(g['opp_team']) for s, g in df_cv.groupby('season')}
ss = sorted(season_teams)
promoted_ever = set().union(*[season_teams[s] - season_teams[ss[i - 1]] for i, s in enumerate(ss) if i > 0])
print('teams promoted at some point in this window:', sorted(promoted_ever))

for arm in ['per_team_sigma', 'per_team_sigma_loose']:
    table = team_sigma_table(arm)
    if table is None:
        continue
    print(f'\n--- {arm}: mean fitted sigma_att_team, highest first ---')
    for team, val in table.items():
        flag = '  <- ever promoted' if team in promoted_ever else ''
        print(f'  {team:<5} {val:.4f}{flag}')
    top_half = set(table.index[: len(table) // 2])
    overlap = len(top_half & promoted_ever)
    print(f'promoted teams in the top half of fitted sigma: {overlap}/{len(promoted_ever & set(table.index))}')

teams promoted at some point in this window: ['BOU', 'BRE', 'BUR', 'FLH', 'IPS', 'LED', 'LEI', 'LUT', 'NOR', 'NOT', 'SHE', 'SOU', 'SUN', 'WAT']

--- per_team_sigma: mean fitted sigma_att_team, highest first ---
  BRI   0.0182
  CHE   0.0162
  ARS   0.0150
  NEW   0.0145
  LIV   0.0128
  FLH   0.0125  <- ever promoted
  MCI   0.0122
  WOL   0.0113
  NOR   0.0113  <- ever promoted
  AVL   0.0111
  SHE   0.0111  <- ever promoted
  WAT   0.0111  <- ever promoted
  SOU   0.0109  <- ever promoted
  WBA   0.0107
  SUN   0.0107  <- ever promoted
  LUT   0.0106  <- ever promoted
  IPS   0.0105  <- ever promoted
  LEI   0.0104  <- ever promoted
  NOT   0.0104  <- ever promoted
  BOU   0.0102  <- ever promoted
  EVE   0.0099
  TOT   0.0097
  LED   0.0096  <- ever promoted
  WHU   0.0096
  BUR   0.0095  <- ever promoted
  MUN   0.0094
  CRY   0.0092
  BRE   0.0091  <- ever promoted
promoted teams in the top half of fitted sigma: 5/14

--- per_team_sigma_loose: mean fitted sigma_att_team, highest f

## Phase 3 — Confirmation (finalist only)

Set `FINALIST` after reading Phase 2, run the full 35-window CV, then the same cold held-out-2025-26-season check WP005 used.

**Heavy compute — run yourself.**

In [ ]:
FINALIST = None   # <- set to 'per_team_sigma' or 'per_team_sigma_loose' after Phase 2

FULL_WINDOWS = list(range(1, len(windows) + 1))
if FINALIST:
    ov = ARMS[FINALIST]
    ckpt = WP006 / f'cv_checkpoint_full_{FINALIST}.pkl'
    done = {r['window'] for r in load_ckpt(ckpt)['results']}
    for w in [x for x in FULL_WINDOWS if x not in done]:
        print(f'[full {FINALIST}] window {w}')
        try:
            subprocess.run([sys.executable, str(SCRIPT), '--data-path', str(DATA_PATH),
                            '--checkpoint-path', str(ckpt), '--window-index', str(w),
                            '--config-json', json.dumps(ov)], timeout=WINDOW_TIMEOUT, check=True)
        except subprocess.SubprocessError as e:
            print(f'  window {w} problem ({type(e).__name__}) — re-run to retry')
    print('finalist full-CV run complete')
else:
    print('set FINALIST first')

### Phase 3 — held-out-season check

In [ ]:
last_season = sorted(df_cv['season'].unique())[-1]
train_end_round = int(df_cv[df_cv['season'] != last_season]['round'].max())
test_rounds = sorted(df_cv[df_cv['season'] == last_season]['round'].unique())
holdout_window = {'train_start': 1, 'train_end': train_end_round,
                  'test_start': int(test_rounds[0]), 'test_end': int(test_rounds[-1])}
print('held-out window:', holdout_window, f'({len(test_rounds)} rounds of {last_season})')

ho_data = WP006 / 'holdout_shared_data.pkl'
pickle.dump({'df_cv': df_cv, 'windows': [holdout_window]}, open(ho_data, 'wb'))

arms_to_run = ['baseline'] + ([FINALIST] if FINALIST else [])
for arm in arms_to_run:
    ov = ARMS[arm]
    ckpt = WP006 / f'cv_checkpoint_holdout_{arm}.pkl'
    if load_ckpt(ckpt)['results']:
        print(f'{arm}: holdout already done'); continue
    print(f'[holdout {arm}]')
    subprocess.run([sys.executable, str(SCRIPT), '--data-path', str(ho_data),
                    '--checkpoint-path', str(ckpt), '--window-index', '1',
                    '--config-json', json.dumps(ov)], timeout=WINDOW_TIMEOUT, check=True)

for arm in arms_to_run:
    ck = WP006 / f'cv_checkpoint_holdout_{arm}.pkl'
    if not load_ckpt(ck)['results']:
        continue
    d = fixtures_for(load_ckpt(ck), windows_list=[holdout_window])
    pin_ok = d[['PSCH', 'PSCD', 'PSCA']].notna().all(axis=1)
    d = d[pin_ok].copy()
    Pin = np.array([devig(r) for r in d[['PSCH', 'PSCD', 'PSCA']].to_numpy()])
    rm = np.array([rps_row(x.p_home, x.p_draw, x.p_away, x.result) for x in d.itertuples()])
    rp = np.array([rps_row(Pin[i, 0], Pin[i, 1], Pin[i, 2], d['result'].iloc[i]) for i in range(len(d))])
    g, lo, hi = boot(rm - rp)
    print(f'{arm:>22}  n={len(d)}  model RPS {rm.mean():.4f}  gap vs Pinnacle {g:+.4f}  CI [{lo:+.4f},{hi:+.4f}]')

## Results

See `README.md` for the writeup once Phases 2–3 are run.